# Tarea 8 · Detección de valores atípicos

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/8_outliers.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Julián.

Analizamos la antigüedad, el cargo mensual y el cargo acumulado en `data/processed/telco_step1.csv`. El resultado es una decisión documentada para la tarea 9.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repositorio y ejecutá todas las celdas.
- **Colab:** abrí el botón y ejecutá todas las celdas. El CSV de la tarea 6 se lee desde GitHub. El botón apunta a `main` y abrirá el notebook una vez integrado. Para probarlo antes, subí el archivo `.ipynb` a Colab.

## 1. Carga de los datos

Leemos la salida de la tarea 6. Buscamos el CSV en el repo cuando ejecutamos en local y usamos la URL de GitHub en Colab. Verificamos que las tres variables sean numéricas y estén completas para que los conteos consideren a todos los clientes.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

RUTA_STEP1 = Path("data/processed/telco_step1.csv")
URL_STEP1 = (
    "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    f"{RUTA_STEP1.as_posix()}"
)

if EN_COLAB:
    fuente = URL_STEP1
else:
    actual = Path.cwd().resolve()
    fuente = next(
        (ruta / RUTA_STEP1 for ruta in (actual, *actual.parents)
         if (ruta / RUTA_STEP1).is_file()),
        None,
    )
    if fuente is None:
        raise FileNotFoundError(f"Falta {RUTA_STEP1}. Ejecutá primero la tarea 6.")

variables = ["tenure", "MonthlyCharges", "TotalCharges"]
datos = pd.read_csv(fuente, usecols=variables)[variables]
assert all(pd.api.types.is_numeric_dtype(datos[columna]) for columna in variables)
assert datos.notna().all().all(), "Hay valores faltantes en las variables analizadas."
print(f"Clientes analizados: {len(datos):,}")

## 2. Detección con el rango intercuartílico (IQR)

Q1 es el percentil 25 y Q3 es el percentil 75. La distancia entre ambos (IQR) mide cuánto se extiende el 50 % central de los valores. Para cada variable, marcamos como atípico un valor estrictamente menor que Q1 − 1,5 × IQR o mayor que Q3 + 1,5 × IQR.

In [ ]:
from IPython.display import display

q1 = datos.quantile(0.25)
q3 = datos.quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
atipicos = datos.lt(limite_inferior) | datos.gt(limite_superior)

resumen = pd.DataFrame({
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "Límite inferior": limite_inferior,
    "Límite superior": limite_superior,
    "Mínimo observado": datos.min(),
    "Máximo observado": datos.max(),
    "Valores atípicos": atipicos.sum(),
})
display(resumen.round(3))

Las tres variables tienen **0 valores atípicos** según este criterio. Incluso los cargos acumulados más altos, que llegan a 8.684,80, están dentro del límite superior de 8.868,675. Los 11 ceros imputados en la tarea 6 también quedan dentro de los límites.

## 3. Boxplots

Mostramos cada variable en su propia escala. La caja va de Q1 a Q3 y la línea interior marca la mediana. Los bigotes llegan hasta los valores observados más alejados dentro de los límites del IQR. Los puntos fuera de los bigotes se mostrarían por separado.

In [ ]:
import matplotlib.pyplot as plt

etiquetas = ["Antigüedad (meses)", "Cargo mensual", "Cargo acumulado"]
fig, ejes = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")
for eje, variable, etiqueta in zip(ejes, variables, etiquetas):
    eje.boxplot(datos[variable], vert=False, whis=1.5, showfliers=True)
    eje.set_title(variable)
    eje.set_xlabel(etiqueta)
    eje.set_yticks([])

fig.suptitle("Distribución de antigüedad y cargos")
plt.show()

Los tres gráficos muestran distribuciones sin puntos fuera de los bigotes. En `TotalCharges`, el bigote hacia los importes altos es más largo, algo compatible con que algunos clientes acumulen cargos durante más meses o paguen más por mes. La amplitud de ese tramo describe diferencias de facturación entre clientes.

## 4. Decisión para la tarea 9

**Conservar los 7.043 registros y sus valores.** El criterio de 1,5 × IQR no detectó valores atípicos en ninguna de las tres variables. Para la tarea 9 no corresponde recortar cargos ni excluir clientes por este análisis. La validez y consistencia de los datos se revisan por separado en esa tarea.